In [5]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 9.1 MB/s eta 0:00:00


In [8]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Define the structure
model = DiscreteBayesianNetwork([
    ('Rain', 'Sprinkler'),
    ('Rain', 'GrassWet'),
    ('Sprinkler', 'GrassWet')
])

# P(Rain)
cpd_rain = TabularCPD(
    variable='Rain',
    variable_card=2,
    values=[[0.2], [0.8]]
)

# P(Sprinkler | Rain)
cpd_sprinkler = TabularCPD(
    variable='Sprinkler',
    variable_card=2,
    values=[
        [0.01, 0.4],
        [0.99, 0.6]
    ],
    evidence=['Rain'],
    evidence_card=[2]
)

# P(GrassWet | Rain, Sprinkler)
cpd_grass = TabularCPD(
    variable='GrassWet',
    variable_card=2,
    values=[
        [0.99, 0.8, 0.9, 0.0],
        [0.01, 0.2, 0.1, 1.0]
    ],
    evidence=['Rain', 'Sprinkler'],
    evidence_card=[2, 2]
)

# Add CPDs
model.add_cpds(cpd_rain, cpd_sprinkler, cpd_grass)

# Check model
model.check_model()

# Inference
inference = VariableElimination(model)

# Query: P(Rain | GrassWet = True)
result = inference.query(variables=['Rain'], evidence={'GrassWet': 0})

print(result)

+---------+-------------+
| Rain    |   phi(Rain) |
+=========+=============+
| Rain(0) |      0.3577 |
+---------+-------------+
| Rain(1) |      0.6423 |
+---------+-------------+
